# Objectif

Construire le Data Warehouse dans PostgreSQL.  
**Pertinence dans le mémoire**

Cette étape matérialise :
- le système décisionnel ;
- l’architecture BI ;
- le modèle multidimensionnel.

## 1. Connexion PostgreSQL

- SQLAlchemy ;
- psycopg2.


In [8]:
# IMPORTATION DES LIBRAIRIES
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sqlalchemy import text

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

print("Librairies chargées")

Librairies chargées


In [3]:
# PARAMÈTRES DE CONNEXION
USER = "postgres"

PASSWORD = "keren"

HOST = "localhost"

PORT = "5432"

DATABASE = "btp_sinistres_dwh"

In [9]:
# CONNEXION POSTGRESQL
engine = create_engine(

    f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"

)

print("Connexion PostgreSQL réussie")

Connexion PostgreSQL réussie


In [10]:
# TEST CONNEXION
with engine.connect() as conn:

    result = conn.execute(
        text("SELECT version();")
    )

    for row in result:
        print(row)

('PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35226, 64-bit',)


In [11]:
# CHARGEMENT DES DONNÉES
dataset = pd.read_csv(
    "../data/processed/dataset_analytique.csv"
)

trc = pd.read_csv(
    "../data/processed/sinistre_trc_enrichi.csv"
)

print("Datasets chargés")

Datasets chargés


In [12]:
# APERÇU DES DONNÉES
display(dataset.head())

display(dataset.shape)

,id_sinistre,annee,mois,date_accident,immatriculation,type_engin_sinistre,marque_sinistre,age_vehicule_ans_sinistre,etat_vehicule,departement_accident,axe_routier,type_accident,responsabilite,gravite,tiers_implique,assureur_sinistre,montant_declare_fcfa,franchise_fcfa,montant_indemnise_fcfa,delai_declaration_jours,delai_reglement_jours,statut_dossier,conducteur_anciennete_ans,type_sinistre,type_engin_parc,marque_parc,annee_mise_en_service,age_vehicule_ans_parc,departement_affectation,km_parcourus_estimes,etat_general,assureur_parc,cout_total_fcfa,classe_age,niveau_experience,score_delai,niveau_risque,sinistre_grave,frequence_sinistre,score_risque,tranche_cout,annee_accident,mois_accident,jour_semaine,est_weekend,categorie_km
0,AUTO-2022-001,2022,7,2022-07-17,PRT-0108-BJ,NIVELEUSE,LIEBHERR,13,MAUVAIS,OUÉMÉ,RN6 NATITINGOU-KÉROU,COLLISION FRONTALE,RESPONSABLE,MATÉRIEL GRAVE,OUI,COLINA,1745000,339000,1101000.0,7,119.0,RÉGLÉ,17,AUTO,NIVELEUSE,LIEBHERR,2011,13,ATLANTIQUE,504000,MAUVAIS,COLINA,1440000.0,ANCIEN,EXPERIMENTE,126.0,MOYEN,0,1,93.46,MOYEN,2022,7,Sunday,True,ELEVE
1,AUTO-2022-002,2022,3,2022-03-02,PRT-0068-BJ,BULLDOZER,LIEBHERR,10,MAUVAIS,ALIBORI,VOIRIE URBAINE PARAKOU,RENVERSEMENT,RESPONSABLE,MATÉRIEL LÉGER,OUI,SAAR-BÉNIN,242000,114000,0.0,5,0.0,EN COURS,19,AUTO,BULLDOZER,LIEBHERR,2014,10,OUÉMÉ,250000,MAUVAIS,SAAR-BÉNIN,114000.0,ANCIEN,EXPERIMENTE,5.0,FAIBLE,0,1,10.46,FAIBLE,2022,3,Wednesday,False,ELEVE
2,AUTO-2022-003,2022,9,2022-09-15,PRT-0318-BJ,CAMION BENNE,MAN,9,MAUVAIS,ATLANTIQUE,RN7 COTONOU-OUIDAH,ACCROCHAGE MINEUR,RESPONSABILITÉ PARTAGÉE,MATÉRIEL LÉGER,OUI,COLINA,471000,270000,0.0,5,0.0,EN EXPERTISE,10,AUTO,CAMION BENNE,MAN,2015,9,ATLANTIQUE,295000,MAUVAIS,COLINA,270000.0,ANCIEN,EXPERIMENTE,5.0,FAIBLE,0,2,11.76,FAIBLE,2022,9,Thursday,False,ELEVE
3,AUTO-2022-004,2022,8,2022-08-27,PRT-0586-BJ,NIVELEUSE,VOLVO CE,4,BON,BORGOU,VOIRIE URBAINE PARAKOU,ACCROCHAGE MINEUR,RESPONSABLE,MATÉRIEL GRAVE,NON,NSIA BÉNIN,6000000,321000,4851000.0,2,44.0,RÉGLÉ,5,AUTO,NIVELEUSE,VOLVO CE,2020,4,ATLANTIQUE,148000,BON,NSIA BÉNIN,5172000.0,MOYEN,INTERMEDIAIRE,46.0,ELEVE,0,3,34.64,ELEVE,2022,8,Saturday,True,MOYEN
4,AUTO-2022-005,2022,9,2022-09-27,PRT-0395-BJ,CAMION MALAXEUR BÉTON,DAF,2,BON,OUÉMÉ,PISTE RURALE,SORTIE DE ROUTE,RESPONSABILITÉ PARTAGÉE,MATÉRIEL LÉGER,NON,COLINA,500000,243000,210000.0,9,87.0,RÉGLÉ,17,AUTO,CAMION MALAXEUR BÉTON,DAF,2022,2,LITTORAL,80000,BON,COLINA,453000.0,RECENT,EXPERIMENTE,96.0,FAIBLE,0,2,62.09,FAIBLE,2022,9,Tuesday,False,MOYEN


(410, 46)

In [13]:
# TRUNCATE TABLE FACT
with engine.begin() as conn:

    conn.execute(
        text("""
            TRUNCATE TABLE fact_sinistres CASCADE;
        """)
    )

print("fact_sinistres vidée")

fact_sinistres vidée


In [14]:
# CHARGEMENT DIM_DATE EXISTANTE
dim_date_sql = pd.read_sql(

    "SELECT * FROM dim_date",

    engine

)

display(dim_date_sql.head())

,date_key,date_complete,jour,mois,trimestre,semestre,annee,nom_mois,nom_jour,jour_semaine_num,est_weekend,semaine_annee,est_jour_ferie,est_fin_mois


## 2. Création des dimensions

- dim_date ;
- dim_vehicule ;
- dim_conducteur ;
- dim_chantier ;
- dim_assureur ;
- dim_localisation ;
- dim_type_sinistre.

In [15]:
# CRÉATION DIM_DATE TEMPORAIRE
dim_date = pd.DataFrame()

dim_date["date_complete"] = pd.to_datetime(
    dataset["date_accident"]
)

dim_date["jour"] = dim_date[
    "date_complete"
].dt.day

dim_date["mois"] = dim_date[
    "date_complete"
].dt.month

dim_date["trimestre"] = dim_date[
    "date_complete"
].dt.quarter

dim_date["semestre"] = np.where(

    dim_date["mois"] <= 6,

    1,

    2

)

dim_date["annee"] = dim_date[
    "date_complete"
].dt.year

dim_date["nom_mois"] = dim_date[
    "date_complete"
].dt.month_name()

dim_date["nom_jour"] = dim_date[
    "date_complete"
].dt.day_name()

dim_date["jour_semaine_num"] = dim_date[
    "date_complete"
].dt.weekday

dim_date["est_weekend"] = dim_date[
    "jour_semaine_num"
].isin([5,6])

dim_date = dim_date.drop_duplicates()

In [40]:
# IDENTIFICATION NOUVELLES DATES
nouvelles_dates = dim_date[

    ~dim_date["date_complete"].isin(
        dim_date_sql["date_complete"]
    )

]

print(nouvelles_dates.shape)

(343, 10)


In [41]:
# INSERTION DIM_DATE
if len(nouvelles_dates) > 0:

    nouvelles_dates.to_sql(

        "dim_date",

        engine,

        if_exists="append",

        index=False

    )

    print("Nouvelles dates insérées")

else:

    print("Aucune nouvelle date")

IntegrityError: (psycopg2.errors.UniqueViolation) ERREUR:  la valeur d'une clé dupliquée rompt la contrainte unique « dim_date_date_complete_key »
DETAIL:  La clé « (date_complete)=(2022-07-17) » existe déjà.

[SQL: INSERT INTO dim_date (date_complete, jour, mois, trimestre, semestre, annee, nom_mois, nom_jour, jour_semaine_num, est_weekend) VALUES (%(date_complete__0)s, %(jour__0)s, %(mois__0)s, %(trimestre__0)s, %(semestre__0)s, %(annee__0)s, %(nom_mois__0)s,  ... 66597 characters truncated ... annee__342)s, %(nom_mois__342)s, %(nom_jour__342)s, %(jour_semaine_num__342)s, %(est_weekend__342)s)]
[parameters: {'annee__0': 2022, 'jour_semaine_num__0': 6, 'semestre__0': 2, 'mois__0': 7, 'nom_mois__0': 'July', 'trimestre__0': 3, 'jour__0': 17, 'nom_jour__0': 'Sunday', 'est_weekend__0': True, 'date_complete__0': datetime.datetime(2022, 7, 17, 0, 0), 'annee__1': 2022, 'jour_semaine_num__1': 2, 'semestre__1': 1, 'mois__1': 3, 'nom_mois__1': 'March', 'trimestre__1': 1, 'jour__1': 2, 'nom_jour

In [19]:
dim_vehicule = dataset[

    [
        "immatriculation",
        "type_engin_parc",
        "marque_parc",
        "annee_mise_en_service",
        "age_vehicule_ans_parc",
        "km_parcourus_estimes",
        "etat_general",
        "departement_affectation",
        "classe_age",
        "categorie_km"
    ]

].drop_duplicates()

dim_vehicule.columns = [

    "immatriculation",
    "type_engin",
    "marque",
    "annee_mise_en_service",
    "age_vehicule_ans",
    "km_parcourus_estimes",
    "etat_general",
    "departement_affectation",
    "classe_age",
    "categorie_km"
]

In [20]:
vehicule_sql = pd.read_sql(

    """
    SELECT immatriculation
    FROM dim_vehicule
    """,

    engine

)

In [21]:
nouveaux_vehicules = dim_vehicule[

    ~dim_vehicule["immatriculation"].isin(
        vehicule_sql["immatriculation"]
    )

]

print(nouveaux_vehicules.shape)

(325, 10)


In [22]:
if len(nouveaux_vehicules) > 0:

    nouveaux_vehicules.to_sql(

        "dim_vehicule",

        engine,

        if_exists="append",

        index=False

    )

    print("Nouveaux véhicules insérés")

else:

    print("Aucun nouveau véhicule")

Nouveaux véhicules insérés


In [23]:
dim_conducteur = dataset[

    [
        "conducteur_anciennete_ans",
        "niveau_experience"
    ]

].drop_duplicates()

dim_conducteur["conducteur_id"] = range(

    1,

    len(dim_conducteur) + 1

)

dim_conducteur["tranche_age_conducteur"] = np.where(

    dim_conducteur["conducteur_anciennete_ans"] <= 2,

    "JEUNE",

    "EXPERIMENTE"

)

In [24]:
dim_conducteur.to_sql(

    "dim_conducteur",

    engine,

    if_exists="append",

    index=False

)

print("dim_conducteur alimentée")

dim_conducteur alimentée


In [26]:
dim_assureur = dataset[
    ["assureur"]
].drop_duplicates()

dim_assureur["code_assureur"] = range(

    1,

    len(dim_assureur) + 1

)

dim_assureur["type_contrat"] = "STANDARD"

KeyError: "None of [Index(['assureur'], dtype='object')] are in the [columns]"

In [27]:
assureur_sql = pd.read_sql(

    """
    SELECT assureur
    FROM dim_assureur
    """,

    engine

)

In [28]:
nouveaux_assureurs = dim_assureur[

    ~dim_assureur["assureur"].isin(
        assureur_sql["assureur"]
    )

]

NameError: name 'dim_assureur' is not defined

In [29]:
if len(nouveaux_assureurs) > 0:

    nouveaux_assureurs.to_sql(

        "dim_assureur",

        engine,

        if_exists="append",

        index=False

    )

    print("Assureurs insérés")

else:

    print("Aucun nouvel assureur")

NameError: name 'nouveaux_assureurs' is not defined

In [30]:
dim_localisation = dataset[

    [
        "departement_accident",
        "axe_routier"
    ]

].drop_duplicates()

dim_localisation.columns = [

    "departement",
    "axe_routier"
]

dim_localisation["nom_departement"] = dim_localisation[
    "departement"
]

dim_localisation["region"] = "BENIN"

dim_localisation["type_zone"] = "URBAINE"

In [31]:
dim_localisation.to_sql(

    "dim_localisation",

    engine,

    if_exists="append",

    index=False

)

print("dim_localisation alimentée")

dim_localisation alimentée


In [32]:
dim_type_sinistre = dataset[

    [
        "type_sinistre",
        "type_accident",
        "responsabilite",
        "tiers_implique",
        "gravite",
        "statut_dossier"
    ]

].drop_duplicates()

dim_type_sinistre["nature_sinistre"] = "AUTO"

## 3. Création de fact_sinistres

In [33]:
dim_type_sinistre.to_sql(

    "dim_type_sinistre",

    engine,

    if_exists="append",

    index=False

)

print("dim_type_sinistre alimentée")

DataError: (psycopg2.errors.InvalidTextRepresentation) ERREUR:  syntaxe en entrée invalide pour le type boolean : « OUI »
LINE 1: ...UES ('AUTO', 'COLLISION FRONTALE', 'RESPONSABLE', 'OUI', 'MA...
                                                             ^

[SQL: INSERT INTO dim_type_sinistre (type_sinistre, type_accident, responsabilite, tiers_implique, gravite, statut_dossier, nature_sinistre) VALUES (%(type_sinistre__0)s, %(type_accident__0)s, %(responsabilite__0)s, %(tiers_implique__0)s, %(gravite__0)s, % ... 34510 characters truncated ... 209)s, %(tiers_implique__209)s, %(gravite__209)s, %(statut_dossier__209)s, %(nature_sinistre__209)s)]
[parameters: {'statut_dossier__0': 'RÉGLÉ', 'gravite__0': 'MATÉRIEL GRAVE', 'type_accident__0': 'COLLISION FRONTALE', 'nature_sinistre__0': 'AUTO', 'responsabilite__0': 'RESPONSABLE', 'tiers_implique__0': 'OUI', 'type_sinistre__0': 'AUTO', 'statut_dossier__1': 'EN COURS', 'gravite__1': 'MATÉRIEL LÉGER', 'type_accident__1': 'RENVERSEMENT', 'nat

In [34]:
date_sql = pd.read_sql(
    "SELECT * FROM dim_date",
    engine
)

vehicule_sql = pd.read_sql(
    "SELECT * FROM dim_vehicule",
    engine
)

conducteur_sql = pd.read_sql(
    "SELECT * FROM dim_conducteur",
    engine
)

assureur_sql = pd.read_sql(
    "SELECT * FROM dim_assureur",
    engine
)

localisation_sql = pd.read_sql(
    "SELECT * FROM dim_localisation",
    engine
)

type_sinistre_sql = pd.read_sql(
    "SELECT * FROM dim_type_sinistre",
    engine
)

In [35]:
fact = dataset.copy()

fact["date_complete"] = pd.to_datetime(
    fact["date_accident"]
)

In [36]:
fact = fact.merge(

    date_sql[
        [
            "date_key",
            "date_complete"
        ]
    ],

    on="date_complete",

    how="left"

)

ValueError: You are trying to merge on datetime64[ns] and object columns for key 'date_complete'. If you wish to proceed you should use pd.concat

In [37]:
fact = fact.merge(

    vehicule_sql[
        [
            "vehicule_key",
            "immatriculation"
        ]
    ],

    on="immatriculation",

    how="left"

)

In [38]:
fact = fact.merge(

    conducteur_sql[
        [
            "conducteur_key",
            "conducteur_anciennete_ans"
        ]
    ],

    on="conducteur_anciennete_ans",

    how="left"

)

In [39]:
fact = fact.merge(

    assureur_sql[
        [
            "assureur_key",
            "assureur"
        ]
    ],

    on="assureur",

    how="left"

)

KeyError: 'assureur'

## 4. Création de fact_predictions

## 5. Injection des données

- to_sql ;
- insertions.